# Understanding CNNs, ResNet-101 & MobileNetV1

This is the project to understand computer vision architecture in AI Academy, covering:

- Convolutional neural networks (CNNs)
- The ResNet-101 bottleneck residual design
- The MobileNetV1 depthwise separable architecture

The notebook runs top to bottom like a real script: no hidden functions to jump around to. Every building block is explained before it's built, so each line of Keras code below it is clear in context.

---

## 1. Setup

Tools we need, and why:

- **TensorFlow / Keras** - builds and trains the neural networks
- **NumPy** - works with arrays of numbers (images are just big grids of numbers)
- **Matplotlib** - draws pictures and charts so we can *see* what's going on

**Random seed:** neural networks start with random numbers and shuffle data randomly. Fixing the seed means "use the same randomness every time," so results are reproducible when the notebook is re-run.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

## 2. The Dataset: Fashion-MNIST

**What it is:**
- 70,000 small grayscale photos of clothing (t-shirts, trousers, shoes, bags, etc.)
- Each labeled with one of 10 categories
- The "hello world" dataset for image classification - small enough to train quickly, real enough to be meaningful

**What an image actually is to a computer:**
- A photo isn't stored as a picture, it's stored as a grid of numbers
- Each image here is 28 pixels tall and 28 pixels wide
- Since it's grayscale (not color), each pixel is a single number from **0 (pure black)** to **255 (pure white)**
- Everything a CNN does is really just math performed on this grid of numbers

In [ ]:
(x_train, y_train), (x_test, y_test) = (
    keras.datasets.fashion_mnist.load_data())

print("Training images:", x_train.shape)
print("Test images:    ", x_test.shape)
print("Each image is a", x_train.shape[1], "x", x_train.shape[2],
      "grid of pixel values")

Here are the first 10 training images and their class labels:

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

plt.figure(figsize=(10, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(class_names[y_train[i]], fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()

And here's what one of those images actually looks like as raw numbers - this is literally what the network "sees." Notice the 0s around the edges (empty black background) and the larger numbers in the middle (the bright parts of the garment):

In [ ]:
sample_image = x_train[0]
sample_label = y_train[0]

print(f"Label: {sample_label} ({class_names[sample_label]})")
print("Pixel values (28 x 28):")
for row in sample_image:
    print(" ".join(f"{pixel:3}" for pixel in row))

In [ ]:
plt.imshow(sample_image, cmap='gray')
plt.title(f"Label: {sample_label} ({class_names[sample_label]})")
plt.colorbar(label='Pixel intensity')
plt.show()

**Why we normalize:**
- Raw pixel values run from 0-255
- Neural networks train more smoothly when input numbers are small and consistent (roughly between 0 and 1)
- So we divide every pixel by 255 - this doesn't change what the image *looks like*, it just rescales the numbers so the math behaves better

**Why we add a "channel" dimension:**
- A color photo has 3 numbers per pixel (Red, Green, Blue)
- A grayscale photo only has 1
- Keras's convolution layers always expect that channel number to be explicitly present in the shape, so a `(28, 28)` image becomes `(28, 28, 1)`

In [ ]:
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm = x_test.astype('float32') / 255.0

x_train_norm = x_train_norm[..., np.newaxis]
x_test_norm = x_test_norm[..., np.newaxis]

print("Shape after adding the channel dimension:", x_train_norm.shape)

**One more transformation - one-hot encoding the labels:**
- Right now a label is just a single number, e.g. `3` for "Dress"
- We're going to use a loss function later (`categorical_crossentropy`) that instead expects each label as a list of 10 numbers, all zeros except a single 1 in the position of the correct class
- e.g. label `3` becomes `[0,0,0,1,0,0,0,0,0,0]`
- This is called **one-hot encoding**

In [ ]:
y_train_onehot = keras.utils.to_categorical(y_train, num_classes=10)
y_test_onehot = keras.utils.to_categorical(y_test, num_classes=10)

print("Original label:", y_train[0])
print("One-hot label: ", y_train_onehot[0])

**Splitting off a validation set:**
- We hold back the last 10,000 training images and never let the model train on them
- During training we'll check the model's accuracy on this set after every epoch
- This tells us how well the model is generalizing to images it hasn't memorized, rather than just how well it's memorizing the training set

In [ ]:
x_train_split = x_train_norm[:50000]
y_train_split = y_train_onehot[:50000]
x_val_split = x_train_norm[50000:]
y_val_split = y_train_onehot[50000:]

print("Training on:  ", x_train_split.shape[0], "images")
print("Validating on:", x_val_split.shape[0], "images")

---

## 3. Building a CNN, Layer by Layer

A **Convolutional Neural Network (CNN)** is the standard architecture for image tasks. Before writing any code, here's what every piece actually does.

### What is a Convolution?
- Think of it as sliding a small magnifying glass over the image, one small patch at a time
- At every location it asks the same question: *"does this specific pattern show up here?"*
- It doesn't look at the whole image at once, it looks at small local neighborhoods
- This is exactly how a human eye picks out edges and shapes before recognizing the whole object

### What is a Kernel / Filter?
- **Kernel** and **filter** mean the same thing here: a small grid of numbers (e.g. 3×3) that the convolution slides across the image
- These numbers start out random and get adjusted during training
- Eventually the filter reliably "lights up" whenever it sees a particular pattern - a vertical edge, a curve, a corner, a texture
- A layer with **16 filters** learns 16 different pattern detectors at once, each producing its own output grid (called a **feature map**)

### Kernel Size
- How big the sliding patch is
- A `3×3` kernel looks at a 3-pixel by 3-pixel neighborhood at each step
- Bigger kernels see more context per step but cost more to compute

### Stride
- How far the kernel jumps between steps
- **Stride 1** slides one pixel at a time (thorough, output stays close to the input size)
- **Stride 2** skips every other position, cutting the output's height and width roughly in half - a cheap way to shrink the image while scanning it

### Padding
- What happens at the edges
- **`'valid'`** padding adds nothing - the output shrinks slightly because the kernel can't center itself right at the border
- **`'same'`** padding adds a border of zeros around the image so the output stays the same width and height as the input

### Activation Function: ReLU
- After a convolution, we apply an **activation function**
- **ReLU** (Rectified Linear Unit) is the simplest common choice: keep positive numbers as they are, turn every negative number into 0
- This sounds trivial, but it's what lets the network learn *non-linear* patterns - without it, stacking many layers would mathematically collapse into the same power as a single layer

### Pooling: Max vs Average
- **Pooling** shrinks the feature maps by summarizing small regions into a single number
- **Max pooling** keeps the strongest signal in each region ("was this pattern detected *anywhere* in this patch?")
- **Average pooling** keeps the average instead ("how much of this pattern was present, on average, in this patch?")
- Pooling makes the network faster and a little more tolerant of the exact pixel position of a pattern

### Flatten
- After several convolution + pooling layers, we're left with a small 3D grid of numbers
- **Flatten** just lays that grid out as one long 1D list, so it can be fed into a standard fully connected layer

### Dense (Fully Connected) Layer
- A **Dense** layer connects *every* input number to *every* output neuron
- This is where the network combines everything it's detected across the whole image to make a final decision

### Softmax
- Our final Dense layer has 10 outputs - one score per clothing category
- **Softmax** converts those raw scores into probabilities that add up to 100%
- So the output reads like "92% Sneaker, 5% Sandal, 3% everything else"

Now let's actually build this. We'll use 2 convolution+pooling blocks, growing the filter count from 16 to 32 (more filters deeper in the network = room to combine simple patterns into more complex ones):

In [ ]:
# ---- Input: one 28x28 grayscale image at a time ----
cnn_inputs = keras.Input(shape=(28, 28, 1), name="input_image")

# ---- Block 1: 16 filters, each a 3x3 pattern detector ----
x = keras.layers.Conv2D(
    filters=16,
    kernel_size=(3, 3),
    strides=1,          # slide one pixel at a time
    padding="valid",    # no padding -> output shrinks: 28x28 -> 26x26
    activation="relu",  # zero out negative values after the convolution
    name="conv2d_1",
)(cnn_inputs)

x = keras.layers.MaxPooling2D(
    pool_size=(2, 2),   # look at 2x2 blocks
    name="max_pool_1",  # keeps the strongest value in each block
)(x)                    # halves height & width: 26x26 -> 13x13

# ---- Block 2: 32 filters -> more filters = more complex patterns ----
x = keras.layers.Conv2D(
    filters=32,
    kernel_size=(3, 3),
    strides=1,
    padding="valid",
    activation="relu",
    name="conv2d_2",
)(x)                    # 13x13 -> 11x11

x = keras.layers.MaxPooling2D(
    pool_size=(2, 2), name="max_pool_2",
)(x)                    # 11x11 -> 5x5

# ---- Flatten the 5x5x32 grid into one long list of numbers ----
x = keras.layers.Flatten(name="flatten")(x)

# ---- Final decision layer: 10 scores -> softmax turns them into percentages ----
cnn_outputs = keras.layers.Dense(
    10, activation="softmax", name="predictions",
)(x)

cnn_model = keras.Model(
    inputs=cnn_inputs, outputs=cnn_outputs, name="fashion_mnist_cnn")
cnn_model.summary()

**Seeing what a filter actually detects.** All the talk about "pattern detectors" is easier to trust once you actually see it. Below we peek inside the model at the output of the first convolution layer (`conv2d_1`) for one real image - each small panel is one of the 16 filters' feature maps. Some will highlight edges, some will highlight the garment's outline, some may look like noise this early in training:

In [ ]:
# Build a small "peek" model that stops right after the first conv layer
feature_map_model = keras.Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer("conv2d_1").output,
)

sample_input = x_train_norm[0:1]   # one image, with batch dimension
feature_maps = feature_map_model.predict(sample_input, verbose=0)

print("Feature map shape:", feature_maps.shape)  # (1, 26, 26, 16) -> 16 filters

plt.figure(figsize=(12, 6))
for filter_index in range(16):
    plt.subplot(2, 8, filter_index + 1)
    plt.imshow(feature_maps[0, :, :, filter_index], cmap="viridis")
    plt.title(f"Filter {filter_index}", fontsize=8)
    plt.axis("off")
plt.suptitle("What each of the 16 filters in conv2d_1 detected")
plt.tight_layout()
plt.show()

---

## 4. Compiling & Training: Loss, Optimizers, Epochs, Batches

A freshly built model is just random numbers, it hasn't learned anything yet. **Training** is the process of showing it labeled examples and adjusting its internal numbers (weights) so its predictions get closer to being correct.

### What is a Loss Function?
- The **loss function** measures how wrong a prediction is
- High loss = bad prediction, low loss = good prediction
- Training is literally the process of trying to make this number as small as possible

Which loss to use depends on your label format:
- **`categorical_crossentropy`** - use when labels are **one-hot encoded** (like `[0,0,0,1,0,...]`, which is what we made above)
- **`sparse_categorical_crossentropy`** - use instead when labels are plain integers (like `3`), skipping the one-hot step - mathematically equivalent, just a different label format

Since we one-hot encoded our labels earlier, we'll use `categorical_crossentropy` here.

### What is an Optimizer?
The **optimizer** is the algorithm that decides *how* to adjust the weights to reduce the loss, based on the gradient (the direction of steepest improvement):

- **SGD (Stochastic Gradient Descent)** - the simplest approach: take a step in the direction that reduces loss, using one fixed step size (the **learning rate**) for every weight. Simple and reliable, but can be slow and sensitive to that learning rate choice
- **RMSprop** - automatically shrinks the step size for weights that have been changing a lot recently, and grows it for weights that have been barely moving. Smooths out training
- **Adam** - combines RMSprop's per-weight adaptive step size with **momentum** (remembering recent update directions, like a ball rolling downhill building up speed). Usually the best general-purpose default, which is why it's so common

### Epochs
- One **epoch** = one full pass through the entire training dataset
- Training for 5 epochs means the model sees every training image 5 times

### Batch Size
- The model doesn't update its weights after every single image, that would be slow and noisy
- Instead it looks at a small **batch** of images (e.g. 64) at once
- It averages the error across them, and updates weights once per batch

### Verbose
Controls how much progress text gets printed during training:
- `verbose=0` - silent, nothing printed
- `verbose=1` - an animated progress bar with live per-batch updates (the default - great in an interactive notebook, messy when output gets logged to a file)
- `verbose=2` - one clean summary line per epoch. Better for logs and for comparing multiple training runs side by side

In [ ]:
# ---- Compile: attach a loss function, optimizer, and metric to track ----
cnn_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# ---- Train ----
history = cnn_model.fit(
    x_train_split,
    y_train_split,
    validation_data=(x_val_split, y_val_split),
    epochs=5,
    batch_size=64,
    verbose=2,
)

**Watching the loss and accuracy over time.** The `history` object returned by `fit()` keeps every epoch's numbers. Plotting them is the standard way to sanity-check a training run:
- Training and validation lines moving down/up together = healthy learning
- Training improving while validation stalls or worsens = **overfitting** (the model is starting to memorize the training set instead of generalizing)

In [ ]:
epochs_range = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history["loss"], label="Training loss")
plt.plot(epochs_range, history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss over training")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history["accuracy"], label="Training accuracy")
plt.plot(epochs_range, history.history["val_accuracy"],
         label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy over training")
plt.legend()

plt.tight_layout()
plt.show()

**Comparing optimizers side by side.** Same architecture, same data, only the optimizer changes - this shows concretely how much the optimizer choice affects how fast (and how well) a model learns:

In [ ]:
optimizer_options = {
    "Adam": keras.optimizers.Adam(),
    "SGD": keras.optimizers.SGD(learning_rate=0.01),
    "RMSprop": keras.optimizers.RMSprop(),
}

comparison_results = {}

for optimizer_name, optimizer in optimizer_options.items():
    print(f"\n=== Training with {optimizer_name} ===")

    # A fresh, identically-shaped model each time, for a fair comparison
    inputs = keras.Input(shape=(28, 28, 1))
    x = keras.layers.Conv2D(16, (3, 3), activation="relu")(inputs)
    x = keras.layers.MaxPooling2D((2, 2))(x)
    x = keras.layers.Conv2D(32, (3, 3), activation="relu")(x)
    x = keras.layers.MaxPooling2D((2, 2))(x)
    x = keras.layers.Flatten()(x)
    outputs = keras.layers.Dense(10, activation="softmax")(x)
    comparison_model = keras.Model(inputs, outputs)

    comparison_model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    comparison_history = comparison_model.fit(
        x_train_split, y_train_split,
        validation_data=(x_val_split, y_val_split),
        epochs=5, batch_size=64, verbose=2,
    )

    comparison_results[optimizer_name] = (
        comparison_history.history["val_accuracy"][-1])

print("\nFinal validation accuracy by optimizer:")
for name, acc in comparison_results.items():
    print(f"  {name:10s}: {acc:.2%}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(comparison_results.keys(), comparison_results.values(),
        color=["#4C72B0", "#DD8452", "#55A868"])
plt.ylabel("Final validation accuracy")
plt.title("Optimizer comparison after 5 epochs")
plt.ylim(0, 1)
for i, (name, acc) in enumerate(comparison_results.items()):
    plt.text(i, acc + 0.02, f"{acc:.1%}", ha="center")
plt.show()

---

## 5. ResNet & the Bottleneck Block

### The Problem: Very Deep Networks Are Hard to Train
- You might assume "more layers = better"
- In practice, past a certain depth, plain stacks of convolutions get *worse*, not better
- During training, the error signal has to travel backward through every single layer to update the earliest ones
- Picture a game of telephone: a message whispered through 100 people in a line usually comes out garbled by the end
- In a very deep network, that training signal (the **gradient**) tends to shrink to almost nothing by the time it reaches the earliest layers, so they barely learn at all
- This is called the **vanishing gradient problem**

### The Fix: Residual (Skip) Connections
- **ResNet's** key idea: add a "shortcut hallway" around each block that lets the original input skip straight to the output, unchanged, in addition to going through the block's convolutions
- Concretely: `output = block(input) + input`
- Each block only has to learn the *difference* it needs to add - if a block turns out not to be useful for a particular input, it can learn to output close to zero, and the shortcut carries the original signal straight through
- That's a much easier thing for a block to learn than "reproduce the input perfectly from scratch," which is what a plain (non-residual) stack would have to do just to avoid making things worse

### The Bottleneck Design: Why 1×1 to 3×3 to 1×1?
Running a 3×3 convolution directly on a wide input (say 256 channels) is expensive. The **bottleneck** trick:

1. A cheap **1×1 convolution** first squeezes the channel count down (e.g. 256 → 64) - this is called a **projection**, and 1×1 convolutions are cheap because they don't look at any neighboring pixels, only across channels at the same pixel
2. The expensive **3×3 convolution** (the one that actually detects spatial patterns) runs on this smaller, cheaper representation
3. A final **1×1 convolution** expands the channels back up (e.g. 64 → 256), matching what the rest of the network expects

This hourglass shape (wide → narrow → wide) gets most of the representational benefit of a full-width 3×3 convolution at a fraction of the computational cost.

### Batch Normalization
- After each convolution, **Batch Normalization** rescales that layer's outputs to have a consistent, well-behaved range before passing them on
- Think of it like re-centering a set of exam scores before comparing them across different classes - without it, one unusually large value early on can throw off everything that depends on it downstream
- It also lets you safely use higher learning rates and speeds up training

### Matching Shapes: the Projection Shortcut
- The shortcut addition (`block(input) + input`) only works if both sides have the *exact same shape*
- Most blocks satisfy this automatically (an **identity shortcut** - pass the input through unchanged)
- But when a block changes the spatial size (via `stride=2`) or the channel count, the raw input can't be added directly anymore
- So the shortcut itself is passed through its own small 1×1 convolution + Batch Normalization, purely to reshape it to match, not to detect any new patterns

Let's build a single bottleneck block by hand, exactly as it would appear at the start of a ResNet stage - input has 64 channels, and we squeeze down to 16 before expanding back to `16 × 4 = 64`, so the shapes line up and we can use a simple identity shortcut:

In [ ]:
bottleneck_input = keras.Input(shape=(28, 28, 64), name="bottleneck_input")

# ---- Step 1: squeeze 64 channels down to 16 (cheap 1x1 convolution) ----
y = keras.layers.Conv2D(
    16, kernel_size=1, strides=1, use_bias=False, name="reduce_1x1",
)(bottleneck_input)
y = keras.layers.BatchNormalization(name="bn_after_reduce")(y)
y = keras.layers.ReLU(name="relu_after_reduce")(y)

# ---- Step 2: the actual spatial pattern detection, on the cheaper 16-channel version ----
y = keras.layers.Conv2D(
    16, kernel_size=3, strides=1, padding="same",
    use_bias=False, name="spatial_3x3",
)(y)
y = keras.layers.BatchNormalization(name="bn_after_spatial")(y)
y = keras.layers.ReLU(name="relu_after_spatial")(y)

# ---- Step 3: expand back up to 16 * 4 = 64 channels ----
y = keras.layers.Conv2D(
    16 * 4, kernel_size=1, strides=1, use_bias=False, name="expand_1x1",
)(y)
y = keras.layers.BatchNormalization(name="bn_after_expand")(y)

# ---- Shortcut: input already has 64 channels, same as y -> add directly ----
shortcut = bottleneck_input

block_output = keras.layers.Add(name="add_shortcut")([y, shortcut])
block_output = keras.layers.ReLU(name="final_relu")(block_output)

bottleneck_demo = keras.Model(
    bottleneck_input, block_output, name="bottleneck_demo")
bottleneck_demo.summary()

---

## 6. Stacking Bottlenecks into ResNet-101

ResNet-101 is built from **4 stages** of bottleneck blocks, commonly called `conv2_x`, `conv3_x`, `conv4_x`, `conv5_x`, with **3, 4, 23, and 3 blocks** respectively:

| Stage | Blocks | Filters (before ×4 expansion) |
|-------|--------|-------------------------------|
| conv2_x | 3  | 64  |
| conv3_x | 4  | 128 |
| conv4_x | 23 | 256 |
| conv5_x | 3  | 512 |

**Where the name comes from:**
- 33 blocks × 3 convolutions each = 99
- Plus the initial stem convolution = 100
- Plus the final classification layer = **101 weighted layers**
- That's literally where "ResNet-101" gets its name

**Downsampling strategy:**
- Only the *first* block of each stage (except the very first stage) uses `stride=2`, halving the height and width
- Each stage after that also doubles the filter count
- This is a deliberate trade-off: as the image gets spatially smaller, the network compensates by representing more channels - roughly keeping the total amount of computation per stage balanced

**Global Average Pooling** at the end replaces `Flatten` + a large `Dense` layer:
- Instead of turning every value in the final feature map into a separate input to the classifier, it just averages each channel down to a single number
- "How much of this pattern was found, on average, anywhere in the image?"
- This uses dramatically fewer parameters than flattening would

In [ ]:
stage_configs = [
    {"name": "conv2", "blocks": 3,  "filters": 64,  "stride": 1},
    {"name": "conv3", "blocks": 4,  "filters": 128, "stride": 2},
    {"name": "conv4", "blocks": 23, "filters": 256, "stride": 2},
    {"name": "conv5", "blocks": 3,  "filters": 512, "stride": 2},
]

resnet_input = keras.Input(shape=(224, 224, 3), name="resnet_input")

# ---- Stem: one standard convolution + pooling before any residual blocks ----
x = keras.layers.Conv2D(
    64, kernel_size=7, strides=2, padding="same",
    use_bias=False, name="stem_conv",
)(resnet_input)
x = keras.layers.BatchNormalization(name="stem_bn")(x)
x = keras.layers.ReLU(name="stem_relu")(x)
x = keras.layers.MaxPooling2D(
    pool_size=3, strides=2, padding="same", name="stem_maxpool",
)(x)

# ---- Stack every stage's bottleneck blocks ----
for stage in stage_configs:
    for block_index in range(stage["blocks"]):
        block_name = f"{stage['name']}_block{block_index + 1}"

        # Only the first block of a stage downsamples spatially.
        # The very first block of EVERY stage still needs a projection
        # shortcut, since the channel count is always changing there.
        block_stride = stage["stride"] if block_index == 0 else 1
        needs_projection = (block_index == 0)

        shortcut = x

        y = keras.layers.Conv2D(
            stage["filters"], 1, strides=block_stride, use_bias=False,
            name=f"{block_name}_conv1",
        )(x)
        y = keras.layers.BatchNormalization(name=f"{block_name}_bn1")(y)
        y = keras.layers.ReLU(name=f"{block_name}_relu1")(y)

        y = keras.layers.Conv2D(
            stage["filters"], 3, padding="same", use_bias=False,
            name=f"{block_name}_conv2",
        )(y)
        y = keras.layers.BatchNormalization(name=f"{block_name}_bn2")(y)
        y = keras.layers.ReLU(name=f"{block_name}_relu2")(y)

        y = keras.layers.Conv2D(
            stage["filters"] * 4, 1, use_bias=False,
            name=f"{block_name}_conv3",
        )(y)
        y = keras.layers.BatchNormalization(name=f"{block_name}_bn3")(y)

        if needs_projection:
            shortcut = keras.layers.Conv2D(
                stage["filters"] * 4, 1, strides=block_stride,
                use_bias=False, name=f"{block_name}_shortcut_conv",
            )(x)
            shortcut = keras.layers.BatchNormalization(
                name=f"{block_name}_shortcut_bn")(shortcut)

        x = keras.layers.Add(name=f"{block_name}_add")([y, shortcut])
        x = keras.layers.ReLU(name=f"{block_name}_out")(x)

# ---- Classification head ----
x = keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
resnet_outputs = keras.layers.Dense(
    1000, activation="softmax", name="predictions")(x)

resnet101_model = keras.Model(
    resnet_input, resnet_outputs, name="resnet101")

print(f"Total layers:     {len(resnet101_model.layers)}")
print(f"Total parameters: {resnet101_model.count_params():,}")

---

## 7. Depthwise Separable Convolutions: MobileNet's Trick

ResNet-101 is powerful but heavy - great on a server with a GPU, too slow and too large for a phone. **MobileNetV1** was designed specifically to run efficiently on mobile and embedded devices, and its core trick is splitting a standard convolution into two cheaper steps.

### Standard Convolution (what we used above)
A normal convolution does two jobs **at once**:
- Looks at spatial neighborhoods ("is there an edge here?")
- **And** mixes information across every input channel, for every output filter
- That combination is expensive

### Depthwise Convolution
- Applies **one separate 3×3 filter per input channel**
- No mixing across channels at all, purely spatial filtering, channel by channel

### Pointwise Convolution
- A **1×1 convolution** that does the channel-mixing job separately and cheaply
- Only looks at a single pixel position across all channels, no spatial neighborhood

### Why This Is So Much Cheaper: a Concrete Comparison
Take a 3×3 convolution going from 64 input channels to 128 output channels:

| Approach | Calculation | Parameters |
|----------|-------------|------------|
| Standard convolution | 3×3×64×128 | **73,728** |
| Depthwise (3×3×64) + Pointwise (1×1×64×128) | 576 + 8,192 | **8,768** |

That's roughly **8× fewer parameters** (and a similar reduction in computation) for essentially the same shape transformation - which is exactly why MobileNet can run on a phone where a plain ResNet can't.

In [ ]:
mobile_block_input = keras.Input(
    shape=(32, 32, 64), name="mobile_block_input")

# ---- Depthwise: one 3x3 filter PER channel, no channel mixing ----
z = keras.layers.DepthwiseConv2D(
    kernel_size=3, strides=1, padding="same",
    use_bias=False, name="depthwise_conv",
)(mobile_block_input)
z = keras.layers.BatchNormalization(name="depthwise_bn")(z)
z = keras.layers.ReLU(name="depthwise_relu")(z)

# ---- Pointwise: a 1x1 convolution that DOES mix channels ----
z = keras.layers.Conv2D(
    128, kernel_size=1, padding="same",
    use_bias=False, name="pointwise_conv",
)(z)
z = keras.layers.BatchNormalization(name="pointwise_bn")(z)
z = keras.layers.ReLU(name="pointwise_relu")(z)

depthwise_demo = keras.Model(
    mobile_block_input, z, name="depthwise_separable_demo")
depthwise_demo.summary()

---

## 8. The Full MobileNetV1 Network

MobileNetV1's body:
- One standard convolution (the **stem**)
- Then **13 depthwise separable blocks** in a row
- Downsampling (`stride=2`) whenever the channel count steps up
- Ending at a small 7×7 feature map with 1024 channels
- Then the same global-average-pooling + softmax classification head we used for ResNet

In [ ]:
mobilenet_stage_config = [
    (64, 1), (128, 2), (128, 1), (256, 2), (256, 1),
    (512, 2), (512, 1), (512, 1), (512, 1), (512, 1), (512, 1),
    (1024, 2), (1024, 1),
]

mobilenet_input = keras.Input(shape=(224, 224, 3), name="mobilenet_input")

# ---- Stem: one standard convolution ----
x = keras.layers.Conv2D(
    32, 3, strides=2, padding="same", use_bias=False,
    name="mobilenet_stem_conv",
)(mobilenet_input)
x = keras.layers.BatchNormalization(name="mobilenet_stem_bn")(x)
x = keras.layers.ReLU(name="mobilenet_stem_relu")(x)

# ---- 13 depthwise separable blocks ----
for block_index, (filters, stride) in enumerate(mobilenet_stage_config):
    block_name = f"dw_block_{block_index + 1}"

    x = keras.layers.DepthwiseConv2D(
        3, strides=stride, padding="same", use_bias=False,
        name=f"{block_name}_depthwise",
    )(x)
    x = keras.layers.BatchNormalization(
        name=f"{block_name}_depthwise_bn")(x)
    x = keras.layers.ReLU(name=f"{block_name}_depthwise_relu")(x)

    x = keras.layers.Conv2D(
        filters, 1, padding="same", use_bias=False,
        name=f"{block_name}_pointwise",
    )(x)
    x = keras.layers.BatchNormalization(
        name=f"{block_name}_pointwise_bn")(x)
    x = keras.layers.ReLU(name=f"{block_name}_pointwise_relu")(x)

# ---- Classification head ----
x = keras.layers.GlobalAveragePooling2D(
    name="mobilenet_global_avg_pool")(x)
mobilenet_outputs = keras.layers.Dense(
    1000, activation="softmax", name="mobilenet_predictions")(x)

mobilenetv1_model = keras.Model(
    mobilenet_input, mobilenet_outputs, name="mobilenetv1")

print(f"Total parameters: {mobilenetv1_model.count_params():,}")

---

## 9. Putting It Side by Side

Same job (classifying a 224×224 image into 1000 categories), two very different design philosophies:
- **ResNet-101** spends its computational budget on depth and residual connections for maximum accuracy
- **MobileNetV1** spends it on depthwise separable convolutions for maximum efficiency

Run the cell below to see the actual parameter counts side by side.

In [ ]:
print("Model size comparison")
print("-" * 45)
print(f"ResNet-101  : {resnet101_model.count_params():>12,} parameters")
print(f"MobileNetV1 : {mobilenetv1_model.count_params():>12,} parameters")

In [ ]:
model_names = ["ResNet-101", "MobileNetV1"]
param_counts = [
    resnet101_model.count_params(),
    mobilenetv1_model.count_params(),
]

plt.figure(figsize=(6, 4))
bars = plt.bar(model_names, param_counts, color=["#C44E52", "#55A868"])
plt.ylabel("Total parameters")
plt.title("Model size: ResNet-101 vs MobileNetV1")
for bar, count in zip(bars, param_counts):
    plt.text(bar.get_x() + bar.get_width() / 2, count,
             f"{count:,}", ha="center", va="bottom")
plt.show()

**The takeaway:**
- MobileNetV1 typically ends up somewhere around 10× smaller than ResNet-101
- For a real (if smaller) drop in accuracy - the classic *accuracy vs. efficiency* trade-off that shows up constantly in real-world model selection
- A server backend with GPUs can afford ResNet's size for the accuracy gain
- A phone app running offline needs MobileNet's much lighter footprint instead